In [ ]:

# Plantilla Extendida: Aprendizaje No Supervisado con K-Means + Limpieza de Dataset
# (Continuación de la plantilla anterior. Añade esta sección después de "1. Cargar el Dataset")

# 1.5. Limpieza del Dataset (Nueva sección)
# Asume X es un NumPy array o Pandas DataFrame. Si es DataFrame, conviértelo a array al final.
import pandas as pd  # Si no lo tienes, agrégalo

# Convertir a DataFrame para facilidad (si X es array)
if isinstance(X, np.ndarray):
    df = pd.DataFrame(X)
else:
    df = X.copy()  # Asume ya es DataFrame; REEMPLAZA si necesario

print("Forma original del dataset:", df.shape)

# Paso 1: Remover duplicados
df = df.drop_duplicates()
print("Forma después de remover duplicados:", df.shape)

# Paso 2: Manejar valores faltantes (NaNs)
# Opción: Eliminar filas con NaNs (si pocos)
# df = df.dropna()  # Descomenta si quieres eliminar

# Opción recomendada: Imputar con media (para numéricos) o moda (categóricos)
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy='mean')  # O 'median', 'most_frequent'
df_clean = pd.DataFrame(imputer.fit_transform(df), columns=df.columns)
print("Valores faltantes antes:", df.isnull().sum().sum())
print("Valores faltantes después:", df_clean.isnull().sum().sum())

# Paso 3: Detectar y manejar outliers (usando IQR para cada columna)
Q1 = df_clean.quantile(0.25)
Q3 = df_clean.quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Filtrar outliers (mantener solo datos dentro de bounds)
df_no_outliers = df_clean[~((df_clean < lower_bound) | (df_clean > upper_bound)).any(axis=1)]
print("Forma después de remover outliers:", df_no_outliers.shape)

# Opcional: Visualizar outliers antes de remover (boxplot)
plt.figure(figsize=(10, 5))
df_no_outliers.boxplot()
plt.title("Boxplot después de limpieza (sin outliers)")
plt.show()

# Paso 4: Convertir de vuelta a NumPy array para el resto de la plantilla
X_clean = df_no_outliers.values  # REEMPLAZA df_no_outliers con df_clean si no quieres remover outliers

# Actualiza y_true si existe (ajusta índices si removiste filas)
if 'y_true' in locals() and y_true is not None:
    # Asume y_true alineado con X original; filtra igual
    original_indices = df_no_outliers.index  # Si usaste drop, ajusta
    y_true = y_true[original_indices] if len(y_true) == len(X) else y_true

print("Dataset limpio listo. Forma final:", X_clean.shape)

# Ahora, usa X_clean en lugar de X en el resto de la plantilla
# Ej: X_scaled = scaler.fit_transform(X_clean)
# Continúa con la sección 2 de la plantilla original...

In [ ]:
# Plantilla de Código: Aprendizaje No Supervisado con K-Means
# Autor: Grok (basado en tu cuadernillo)
# Fecha: Noviembre 2025
# Instrucciones: Reemplaza los placeholders con tu dataset. Ejecuta paso a paso.

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs  # Para datos sintéticos (ejemplo)
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA  # Para reducción de dimensionalidad si >2 features
from sklearn.metrics import silhouette_score
from sklearn.model_selection import train_test_split  # Opcional para semi-supervisado

# 1. Cargar el Dataset (Reemplaza con tu dataset)
# Ejemplos:
# - Datos sintéticos: X, y = make_blobs(n_samples=2000, centers=5, random_state=42)
# - Iris: from sklearn.datasets import load_iris; data = load_iris(); X = data.data; y_true = data.target (para evaluación)
# - LFW: from sklearn.datasets import fetch_lfw_people; lfw = fetch_lfw_people(); X = lfw.data; y_true = lfw.target
# - CSV personalizado: import pandas as pd; df = pd.read_csv('tu_archivo.csv'); X = df.values

X, y_true = make_blobs(n_samples=2000, centers=5, random_state=42)  # REEMPLAZA AQUÍ con tu X (features) y y_true (etiquetas opcionales para evaluación)

print("Forma de X:", X.shape)  # Verifica dimensiones

# 2. Preprocesamiento: Escalar los datos (K-Means es sensible a escalas)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Opcional: Si X tiene muchas dimensiones (>2), reduce con PCA para visualización
if X_scaled.shape[1] > 2:
    pca = PCA(n_components=2)  # Reduce a 2D
    X_reduced = pca.fit_transform(X_scaled)
    print("Datos reducidos a 2D para visualización.")
else:
    X_reduced = X_scaled  # Si ya es 2D o menos

# 3. Seleccionar el número óptimo de clusters (K) - Método del Codo
inertias = []  # Lista para almacenar la inercia (suma de distancias cuadradas)
siluetas = []  # Lista para silueta scores (opcional)
k_range = range(2, 11)  # Prueba K de 2 a 10 (ajusta según tu dataset)

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42)
    kmeans.fit(X_scaled)
    inertias.append(kmeans.inertia_)  # Inercia
    sil_score = silhouette_score(X_scaled, kmeans.labels_)
    siluetas.append(sil_score)
    print(f"K={k}: Inercia={kmeans.inertia_:.2f}, Silueta={sil_score:.2f}")

# Graficar el método del codo
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.plot(k_range, inertias, 'bo-')
plt.xlabel('Número de Clusters (K)')
plt.ylabel('Inercia')
plt.title('Método del Codo para K Óptimo')

plt.subplot(1, 2, 2)
plt.plot(k_range, siluetas, 'bo-')
plt.xlabel('Número de Clusters (K)')
plt.ylabel('Silueta Score')
plt.title('Silueta Score para K Óptimo')
plt.show()

# Elige K basado en el gráfico (donde la curva se "dobla" o silueta máxima)
optimal_k = 5  # REEMPLAZA AQUÍ con el K que elijas del gráfico

# 4. Aplicar K-Means con K óptimo
kmeans = KMeans(n_clusters=optimal_k, random_state=42)
kmeans.fit(X_scaled)
labels = kmeans.labels_  # Etiquetas de clusters asignadas

# 5. Visualización de Clusters
plt.figure(figsize=(8, 6))
plt.scatter(X_reduced[:, 0], X_reduced[:, 1], c=labels, cmap='viridis', s=50)
plt.scatter(kmeans.cluster_centers_[:, 0] if X_scaled.shape[1] > 2 else kmeans.cluster_centers_[:, 0],
            kmeans.cluster_centers_[:, 1] if X_scaled.shape[1] > 2 else kmeans.cluster_centers_[:, 1],
            s=200, c='red', marker='X', label='Centroides')
plt.xlabel('Feature 1 (o PCA1)')
plt.ylabel('Feature 2 (o PCA2)')
plt.title(f'Clusters con K-Means (K={optimal_k})')
plt.legend()
plt.show()

# 6. Evaluación (si tienes etiquetas verdaderas y_true, opcional)
if 'y_true' in locals() and y_true is not None:
    from sklearn.metrics import adjusted_rand_score
    ari = adjusted_rand_score(y_true, labels)
    print(f"Adjusted Rand Index (comparación con etiquetas verdaderas): {ari:.2f}")

# Silueta general
sil_score_final = silhouette_score(X_scaled, labels)
print(f"Silueta Score Final: {sil_score_final:.2f}")  # Rango: -1 a 1 (mayor es mejor)

# 7. Extensión Opcional: Semi-Supervisado
# Divide en train/test si quieres semi-supervisado
# X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_true, test_size=0.2, random_state=42)
# Luego, usa K-Means en X_train para encontrar representantes:
# dists = kmeans.fit_transform(X_train)
# idxs = np.argmin(dists, axis=0)  # Índices de representantes
# y_representative = y_train[idxs]  # Etiquetas manuales (simuladas)
# Propaga: y_train_propagated = np.empty(len(X_train)); for i in range(optimal_k): y_train_propagated[kmeans.labels_ == i] = y_representative[i]
# Entrena un clasificador: from sklearn.linear_model import LogisticRegression; log_reg = LogisticRegression().fit(X_train, y_train_propagated)

# 8. Extensión Opcional: Aprendizaje Activo
# Después de un modelo inicial, predice probabilidades:
# probas = log_reg.predict_proba(X_train)  # Asume un modelo entrenado
# confidences = np.max(probas, axis=1)
# low_conf_idx = np.argsort(confidences)[:50]  # 50 más inciertas
# # Etiqueta manualmente esas y reentrena

# Fin de la plantilla. ¡Adapta y experimenta!

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, accuracy_score
from sklearn.linear_model import LogisticRegression
# === CARGA Y PREPRO ===
CSV = "dataset.csv"   # <-- cambia el nombre
df  = pd.read_csv(CSV)

# Detectar y (si no existe, comenta estas 3 líneas y usa solo la parte NO supervisada)
posibles_y = [c for c in df.columns if c.lower() in {"y","target","label","class"}]
y = None
if posibles_y:
    y = df.pop(posibles_y[0]).to_numpy()

# Separar tipos de columnas
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in df.columns if c not in num_cols]

# Pipeline de features: OneHot a categóricas + escala numéricas
feat_pipe = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols)
], remainder="drop")

X = feat_pipe.fit_transform(df).astype(np.float32)
print("X shape:", X.shape, "| y:", None if y is None else np.unique(y))
pca95 = PCA(n_components=0.95, random_state=42)
Xr = pca95.fit_transform(X)
print("PCs:", Xr.shape[1], "Var retenida:", pca95.explained_variance_ratio_.sum())
k_min, k_max = 2, 10
scores, inertias, models = [], [], []
for k in range(k_min, k_max+1):
    km = KMeans(n_clusters=k, n_init="auto", random_state=42).fit(Xr)
    lab = km.labels_
    s = silhouette_score(Xr, lab)             # si el dataset es grande: sample_size=3000
    scores.append(s); inertias.append(km.inertia_); models.append((k, km, lab))

scores = np.array(scores); inertias = np.array(inertias)
best_idxs = np.flatnonzero(scores == scores.max())
best_idx  = best_idxs[np.argmin(inertias[best_idxs])]  # desempata por menor SSE
best_k, best_km, best_labels = models[best_idx]
print(f"Mejor k={best_k} | silhouette={scores[best_idx]:.4f}")
pca2 = PCA(n_components=2, random_state=42).fit(X)
X2 = pca2.transform(X)
plt.figure(figsize=(7,4))
plt.scatter(X2[:,0], X2[:,1], c=best_labels, s=6, alpha=.8, linewidths=0)
plt.title(f"K={best_k} proyectado en PCA-2D | PC1={pca2.explained_variance_ratio_[0]*100:.1f}% PC2={pca2.explained_variance_ratio_[1]*100:.1f}%")
plt.xlabel("PC1"); plt.ylabel("PC2"); plt.tight_layout(); plt.show()
